<a href="https://colab.research.google.com/github/zharmo/AdapterDesignPattern/blob/main/ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np

 1. GIVEN VALUES

In [8]:
X = np.array([1.0, 0.5, -1.0, 2.0])
T = np.array([1.0, 0.0])

W1 = np.array([
    [0.1, -0.2, 0.1, 0.2],
    [-0.1, 0.1, 0.2, -0.1]
])

b1 = np.array([0.0, 0.1])

W2 = np.array([
    [0.2, -0.1]
])

b2 = np.array([0.05])

W3 = np.array([
    [0.3],
    [-0.2]
])

b3 = np.array([0.0, 0.1])

learning_rate = 0.5
lambda_reg = 0.1


2. ACTIVATION FUNCTIONS

In [9]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def relu(z):
    return np.maximum(0, z)

 3. FORWARD PROPAGATION

In [10]:
def forward(W1, W2, W3):

    # Hidden Layer 1
    Z1 = W1 @ X + b1

    # h1 = ReLU, h2 = Sigmoid
    A1 = np.array([
        relu(Z1[0]),
        sigmoid(Z1[1])
    ])

    # Hidden Layer 2
    Z2 = W2 @ A1 + b2

    A2 = sigmoid(Z2)

    # Output Layer
    Z3 = W3 @ A2 + b3

    Ypred = sigmoid(Z3)

    return Z1, A1, Z2, A2, Z3, Ypred

4. SQUARED ERROR LOSS

In [11]:
def squared_loss(Ypred):

    return 0.5 * np.sum((Ypred - T) ** 2)


5. L1 AND L2 REGULARIZATION

In [12]:
def L1_loss(W1, W2, W3):

    return lambda_reg * (
        np.sum(np.abs(W1)) +
        np.sum(np.abs(W2)) +
        np.sum(np.abs(W3))
    )


def L2_loss(W1, W2, W3):

    return (lambda_reg / 2) * (
        np.sum(W1 ** 2) +
        np.sum(W2 ** 2) +
        np.sum(W3 ** 2)
    )


6. BACKPROPAGATION

In [13]:
def backpropagation(W1, W2, W3, Z1, A1, A2, Ypred):

    # Output layer error
    delta3 = (Ypred - T) * Ypred * (1 - Ypred)

    # Gradient of W3
    grad_W3 = np.outer(delta3, A2)

    # Hidden layer 2 error
    delta2 = (W3.T @ delta3) * A2 * (1 - A2)

    # Gradient of W2
    grad_W2 = np.outer(delta2, A1)

    # Hidden layer 1 error
    delta1 = (W2.T @ delta2).ravel()

    # Activation derivatives
    activation_derivatives = np.array([
        1.0 if Z1[0] > 0 else 0.0,
        A1[1] * (1 - A1[1])
    ])

    delta1 = delta1 * activation_derivatives

    # Gradient of W1
    grad_W1 = np.outer(delta1, X)

    return grad_W1, grad_W2, grad_W3

 7. INITIAL FORWARD PASS

In [14]:
Z1, A1, Z2, A2, Z3, Ypred = forward(W1, W2, W3)

initial_loss = squared_loss(Ypred)

print("\n========== INITIAL FORWARD PASS ==========")

print("Z1 =", Z1)
print("A1 =", A1)

print("Z2 =", Z2)
print("A2 =", A2)

print("Z3 =", Z3)
print("Ypred =", Ypred)

print("\nSquared Error Loss =", initial_loss)



========== INITIAL FORWARD PASS ==========
Z1 = [ 0.3  -0.35]
A1 = [0.3        0.41338242]
Z2 = [0.06866176]
A2 = [0.5171587]
Z3 = [ 0.15514761 -0.00343174]
Ypred = [0.53870929 0.49914207]

Squared Error Loss = 0.2309659620854817


8. INITIAL REGULARIZATION

In [15]:
initial_L1 = L1_loss(W1, W2, W3)
initial_L2 = L2_loss(W1, W2, W3)

print("\n========== REGULARIZATION ==========")

print("L1 Penalty =", initial_L1)
print("L2 Penalty =", initial_L2)

print("Initial L1 Total Loss =", initial_loss + initial_L1)
print("Initial L2 Total Loss =", initial_loss + initial_L2)


========== REGULARIZATION ==========
L1 Penalty = 0.19000000000000003
L2 Penalty = 0.017500000000000005
Initial L1 Total Loss = 0.42096596208548176
Initial L2 Total Loss = 0.24846596208548172


9. BACKPROPAGATION GRADIENTS

In [19]:
grad_W1, grad_W2, grad_W3 = backpropagation(
    W1, W2, W3, Z1, A1, A2, Ypred
)

print("\n========== BACKPROPAGATION ==========")

print("Gradient W1 w12 =", grad_W1[0, 1])
print("Gradient W2 w11 =", grad_W2[0, 0])
print("Gradient W3 w11 =", grad_W3[0, 0])



========== BACKPROPAGATION ==========
Gradient W1 w12 = -0.001481914532728759
Gradient W2 w11 = -0.004445743598186278
Gradient W3 w11 = -0.059282665101256


10. UPDATE SELECTED WEIGHTS

In [20]:
def update_weights(reg_type):

    # Start with original weights
    new_W1 = W1.copy()
    new_W2 = W2.copy()
    new_W3 = W3.copy()

    selected = [
        ("W1 w12", new_W1, grad_W1, (0, 1)),
        ("W2 w11", new_W2, grad_W2, (0, 0)),
        ("W3 w11", new_W3, grad_W3, (0, 0))
    ]

    print(f"\n========== {reg_type} WEIGHT UPDATE ==========")

    for name, weight, gradient, index in selected:

        old_weight = weight[index]

        data_gradient = gradient[index]

        # L1 gradient
        if reg_type == "L1":
            reg_gradient = lambda_reg * np.sign(old_weight)

        # L2 gradient
        else:
            reg_gradient = lambda_reg * old_weight

        total_gradient = data_gradient + reg_gradient

        new_weight = (
            old_weight -
            learning_rate * total_gradient
        )

        weight[index] = new_weight

        print("\n", name)
        print("Old weight =", old_weight)
        print("Data gradient =", data_gradient)
        print("Regularization gradient =", reg_gradient)
        print("Total gradient =", total_gradient)
        print("New weight =", new_weight)

    return new_W1, new_W2, new_W3


11. NEW FORWARD PASS AND LOSS

In [21]:
def run_experiment(reg_type):

    new_W1, new_W2, new_W3 = update_weights(reg_type)

    # Forward pass using updated weights
    new_Z1, new_A1, new_Z2, new_A2, new_Z3, new_Ypred = (
        forward(new_W1, new_W2, new_W3)
    )

    new_loss = squared_loss(new_Ypred)

    if reg_type == "L1":
        new_penalty = L1_loss(new_W1, new_W2, new_W3)
        old_penalty = initial_L1

    else:
        new_penalty = L2_loss(new_W1, new_W2, new_W3)
        old_penalty = initial_L2

    new_total_loss = new_loss + new_penalty
    old_total_loss = initial_loss + old_penalty

    print(f"\n========== NEW {reg_type} FORWARD PASS ==========")

    print("New Z1 =", new_Z1)
    print("New A1 =", new_A1)

    print("New Z2 =", new_Z2)
    print("New A2 =", new_A2)

    print("New Z3 =", new_Z3)
    print("New Ypred =", new_Ypred)

    print("\nOld Squared Loss =", initial_loss)
    print("New Squared Loss =", new_loss)

    print("Old Total Loss =", old_total_loss)
    print("New Total Loss =", new_total_loss)

    print("\nSquared Loss Change =", new_loss - initial_loss)
    print("Total Loss Change =", new_total_loss - old_total_loss)

    if new_total_loss < old_total_loss:
        print("Regularized objective decreased.")
    else:
        print("Regularized objective increased.")


 12. RUN L1 AND L2 SEPARATELY

In [22]:
run_experiment("L1")

run_experiment("L2")


========== L1 WEIGHT UPDATE ==========

 W1 w12
Old weight = -0.2
Data gradient = -0.001481914532728759
Regularization gradient = -0.1
Total gradient = -0.10148191453272877
New weight = -0.14925904273363563

 W2 w11
Old weight = 0.2
Data gradient = -0.004445743598186278
Regularization gradient = 0.1
Total gradient = 0.09555425640181373
New weight = 0.15222287179909316

 W3 w11
Old weight = 0.3
Data gradient = -0.059282665101256
Regularization gradient = 0.1
Total gradient = 0.040717334898744004
New weight = 0.279641332550628

========== NEW L1 FORWARD PASS ==========
New Z1 = [ 0.32537048 -0.35      ]
New A1 = [0.32537048 0.41338242]
New Z2 = [0.05819059]
New A2 = [0.51454354]
New Z3 = [ 0.14388764 -0.00290871]
New Ypred = [0.53590998 0.49927282]

Old Squared Loss = 0.2309659620854817
New Squared Loss = 0.23232645115590012
Old Total Loss = 0.42096596208548176
New Total Loss = 0.4104387758642358

Squared Loss Change = 0.0013604890704184136
Total Loss Change = -0.010527186221245932
Regu